# 02 · Demo del pipeline de punta a punta
Un evento atraviesa las 10 etapas y produce salidas observables: prioridad, alerta de fraude, recomendaciones Top-K,
consultas por rango, autocompletado y comunidades del grafo.

In [ ]:
import sys, json
import pandas as pd
sys.path.insert(0, '..')
from src.pipeline import Pipeline, cargar_stream
df = cargar_stream('../data/stream_10M.parquet', n=200_000)
bl = pd.read_csv('../data/blacklist.csv', dtype=str)
st = pd.read_csv('../data/catalogo_stock.csv')
p = Pipeline((bl['entity_type'] + ':' + bl['entity']).tolist(),
             dict(zip(st['product_id'], st['stock'])),
             {'refresco_eventos': 50_000})          # los índices se refrescan cada 50K eventos
p.procesar_stream(df)

## Traza de un evento (compra express)

In [ ]:
pd.DataFrame(p.traza)

## Priorización (etapa 2) y filtrado probabilístico (etapa 6)

In [ ]:
r = p.resumen()
print('Preprocesamiento:', r['preprocesamiento'])
print('Atendidos por nivel (0=alerta, 1=express, 2=estándar, 3=programada, 4=carrito/búsqueda, 5=vista):', r['atendidos_por_nivel_heap'])
print('Alertas por motivo:', r['alertas']['por_motivo'])
print(json.dumps(r['filtrado'], indent=1))
pd.DataFrame(r['alertas']['ejemplos'])

## Refresco por micro-lotes: las recomendaciones cambian mientras llega el stream

In [ ]:
pd.DataFrame(r['refrescos_micro_lote'])

## Servicio (etapa 9): recomendaciones, rango de precios y autocompletado

In [ ]:
pid = r['top10_mas_vendidos'][0][0]
p.recomendar(pid)

In [ ]:
prod = p.almacen.producto(pid)
print('Stock disponible:', prod['stock'])
p.productos_por_rango(prod['category_id'], 50, 300)[:20]     # solo productos con stock

In [ ]:
print('Búsquedas más frecuentes:', r['busquedas']['top10'])
{pref: p.autocompletar(pref) for pref in ['sm', 'app', 'sam', 'he']}

## Grafo de co-compra (etapa 8): PageRank vs grado y comunidades

In [ ]:
print('Spearman PageRank vs grado:', r['grafo']['spearman_pagerank_vs_grado'])
pd.DataFrame(r['grafo']['comunidades']).T